# 🚗 Vehicle Loan Default — EDA (Complete Version)

**Framework:** 8 Business × Domain-First Questions  
**Data:** `./data/train.csv` (233,154 rows)  
**Bug fixes applied:** XGBoost datetime, early_stopping_rounds, SMOTE leakage, Silhouette API  
**Additions:** Geographic analysis, time trends, interaction features, threshold tuning, SHAP


## ⚙️ Setup

In [ ]:
import warnings, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

DATA_PATH = "./data/train.csv"
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.upper()

# --- Parse age-string columns (e.g. "2yrs 3mon") → total months ---
def parse_months(s):
    if pd.isna(s): return np.nan
    y = re.search(r"(\d+)yrs", str(s))
    m = re.search(r"(\d+)mon", str(s))
    return int(y.group(1))*12 + int(m.group(1)) if y else np.nan

df["AVERAGE_ACCT_AGE_MON"]    = df["AVERAGE_ACCT_AGE"].apply(parse_months)
df["CREDIT_HISTORY_LEN_MON"]  = df["CREDIT_HISTORY_LENGTH"].apply(parse_months)

# --- Compute age ---
df["DATE_OF_BIRTH"]  = pd.to_datetime(df["DATE_OF_BIRTH"],  dayfirst=True, errors="coerce")
df["DISBURSAL_DATE"] = pd.to_datetime(df["DISBURSAL_DATE"], dayfirst=True, errors="coerce")
df["AGE_AT_DISBURSAL"] = (df["DISBURSAL_DATE"] - df["DATE_OF_BIRTH"]).dt.days / 365

print(df.shape)
print("Default rate:", round(df["LOAN_DEFAULT"].mean(), 4))
df.head(3)


## 🔍 Data Overview

In [ ]:
df.info()


In [ ]:
# Column metadata
meta = pd.DataFrame({
    "dtype"    : df.dtypes,
    "non_null" : df.notnull().sum(),
    "null"     : df.isnull().sum(),
    "null_%"   : (df.isnull().sum() / len(df) * 100).round(2),
    "nunique"  : df.nunique(),
    "sample"   : df.apply(lambda c: c.dropna().iloc[0] if c.notna().any() else np.nan),
})
print("Column Metadata")
display(meta)


In [ ]:
# Missing values
miss = df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)

if miss.empty:
    print("✅ No missing values")
else:
    miss_df = pd.DataFrame({"missing_count": miss, "missing_%": (miss / len(df) * 100).round(2)})
    print("Missing Values:")
    display(miss_df)
    fig, ax = plt.subplots(figsize=(8, 0.5))
    (miss / len(df) * 100).plot(kind="barh", ax=ax, color="tomato")
    ax.set_title("Missing Value Rate (%)")
    ax.set_xlabel("% missing")
    plt.tight_layout(); plt.show()

print(f"\nDuplicates: {df.duplicated().sum()}")


In [ ]:
# Missing: EMPLOYMENT_TYPE
missing_rate = df["EMPLOYMENT_TYPE"].isnull().mean()
emp_miss = df["EMPLOYMENT_TYPE"].isnull().astype(int)
print(f"Missing: {missing_rate:.2%}")
print("Default rate by missing EMPLOYMENT_TYPE:")
print(df.groupby(emp_miss)["LOAN_DEFAULT"].mean())

# Fill missing with 'Unknown'
df["EMPLOYMENT_TYPE"] = df["EMPLOYMENT_TYPE"].fillna("Unknown")
print("\nDefault rate by EMPLOYMENT_TYPE:")
print(df.groupby("EMPLOYMENT_TYPE")["LOAN_DEFAULT"].mean().sort_values(ascending=False))


## 📊 Numeric Summary

In [ ]:
num_cols = df.select_dtypes(include="number").columns.tolist()
num_cols = [c for c in num_cols if c not in ("UNIQUEID", "LOAN_DEFAULT")]
stats = df[num_cols].describe(percentiles=[.25,.5,.75,.95]).T
stats["skew"] = df[num_cols].skew().round(2)
print("Numeric Summary (excl. ID & target):")
display(stats.style.background_gradient(subset=["skew"], cmap="RdYlGn"))


## 🎯 Target Distribution

In [ ]:
vc = df["LOAN_DEFAULT"].value_counts().rename({0:"Not Default", 1:"Default"})
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].bar(vc.index, vc.values, color=["steelblue","tomato"])
for i, v in enumerate(vc.values):
    axes[0].text(i, v + 500, f"{v:,}", ha="center")
axes[0].set_ylabel("Count")
axes[0].set_title("Loan Default — Count")

axes[1].pie(vc.values, labels=vc.index, autopct="%1.1f%%",
            colors=["steelblue","tomato"], startangle=90)
axes[1].set_title("Loan Default — Proportion")

plt.tight_layout(); plt.show()
print(vc.to_string())


## 📈 Numeric Feature Distributions

In [ ]:
key_num = [
    "DISBURSED_AMOUNT","ASSET_COST","LTV","PERFORM_CNS_SCORE",
    "PRI_NO_OF_ACCTS","PRI_ACTIVE_ACCTS","PRI_OVERDUE_ACCTS",
    "AGE_AT_DISBURSAL","AVERAGE_ACCT_AGE_MON","CREDIT_HISTORY_LEN_MON",
    "NO_OF_INQUIRIES","DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS",
]

n_cols = 3
n_rows = -(-len(key_num) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3.5))
axes = axes.flatten()

for i, col in enumerate(key_num):
    axes[i].hist(df[col].dropna(), bins=40, color="steelblue",
                 edgecolor="white", density=True, alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_yticks([])

for i in range(len(key_num), len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Numeric Feature Distributions", y=1.01, fontsize=12)
plt.tight_layout(); plt.show()


## 🔬 Numeric vs Target (Boxplots)

In [ ]:
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*3.5))
axes = axes.flatten()

for i, col in enumerate(key_num):
    sns.boxplot(x="LOAN_DEFAULT", y=col, data=df, ax=axes[i])
    axes[i].set_title(col)

for i in range(len(key_num), len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Numeric Features vs LOAN_DEFAULT", y=1.01, fontsize=12)
plt.tight_layout(); plt.show()


## 🏷️ Categorical Features vs Target

In [ ]:
cat_cols = ["EMPLOYMENT_TYPE", "PERFORM_CNS_SCORE_DESCRIPTION"]

for col in cat_cols:
    vc2 = df[col].value_counts(dropna=False)
    dr = df.groupby(col, dropna=False)["LOAN_DEFAULT"].mean().rename("default_rate")
    summary = pd.concat([vc2, dr], axis=1)
    summary["count_%"] = (summary["count"] / len(df) * 100).round(2)
    print(f"\n--- {col} ---")
    display(summary.sort_values("count", ascending=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, cat_cols):
    dr = (df.pipe(lambda d: d.assign(missing=d[col].fillna("Missing")))
            .groupby("missing")["LOAN_DEFAULT"].mean()
            .sort_values(ascending=False))
    dr.plot(kind="bar", ax=ax, color="tomato", rot=45)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(f"Default Rate by {col}")
    ax.set_ylabel("Default Rate (%)")
    for j, v in enumerate(dr.values):
        ax.text(j, v + 0.003, f"{v:.1%}", ha="center", fontsize=8)

plt.tight_layout(); plt.show()


## 🌡️ Correlation Matrix

In [ ]:
corr_cols = num_cols
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(16, 14))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout(); plt.show()


In [ ]:
# Point-biserial correlation vs target
from scipy.stats import pointbiserialr

results = []
for col in num_cols:
    valid = df[[col, "LOAN_DEFAULT"]].dropna()
    r, pval = pointbiserialr(valid[col].astype(float), valid["LOAN_DEFAULT"])
    results.append({"feature": col, "r": round(r,3), "p_value": round(pval,4)})

corr_df = pd.DataFrame(results).sort_values("r", key=abs, ascending=False)
display(corr_df)

fig, ax = plt.subplots(figsize=(6, 6))
colors = ["tomato" if v > 0 else "steelblue" for v in corr_df["r"].values]
ax.barh(corr_df["feature"], corr_df["r"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Correlation with LOAN_DEFAULT")
ax.set_xlabel("Pearson r")
plt.tight_layout(); plt.show()


## 🗺️ Geographic Analysis (NEW)

Default rate วิเคราะห์ตาม STATE_ID และ CURRENT_PINCODE_ID — feature สำคัญจาก XGBoost importance


In [ ]:
# Default rate by STATE_ID
state_dr = (df.groupby("STATE_ID")["LOAN_DEFAULT"]
              .agg(["mean","count"])
              .rename(columns={"mean":"default_rate","count":"n"})
              .query("n >= 500")
              .sort_values("default_rate", ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
state_dr["default_rate"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_title("Default Rate by STATE_ID (n≥500)")
axes[0].set_xlabel("STATE_ID")

state_dr["n"].plot(kind="bar", ax=axes[1], color="gray", alpha=0.7)
axes[1].set_title("Sample Count by STATE_ID")
axes[1].set_xlabel("STATE_ID")

plt.tight_layout(); plt.show()
display(state_dr.head(10))


In [ ]:
# Top/Bottom PIN codes by default rate
pin_dr = (df.groupby("CURRENT_PINCODE_ID")["LOAN_DEFAULT"]
            .agg(["mean","count"])
            .rename(columns={"mean":"default_rate","count":"n"})
            .query("n >= 100")
            .sort_values("default_rate", ascending=False))

print("Top 10 high-risk pincodes:")
display(pin_dr.head(10))
print("\nTop 10 low-risk pincodes:")
display(pin_dr.tail(10))


## 📅 Time Trend Analysis (NEW)

Default rate เปลี่ยนแปลงตาม DISBURSAL_DATE หรือไม่?


In [ ]:
df["DISBURSAL_YEAR"]  = df["DISBURSAL_DATE"].dt.year
df["DISBURSAL_MONTH"] = df["DISBURSAL_DATE"].dt.to_period("M")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# By year
yr_dr = df.groupby("DISBURSAL_YEAR")["LOAN_DEFAULT"].mean()
yr_dr.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[0].set_title("Default Rate by Disbursal Year")
axes[0].set_ylabel("Default Rate")

# By month (rolling)
mo_dr = df.groupby("DISBURSAL_MONTH")["LOAN_DEFAULT"].mean()
mo_dr.plot(ax=axes[1], color="steelblue")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Default Rate by Disbursal Month")
axes[1].set_xlabel("Month")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()


## 🔀 Interaction Features (NEW)

สร้าง features ที่มีความหมายทางธุรกิจจาก domain knowledge


In [ ]:
# 1. Primary overdue ratio
df["PRI_OVERDUE_RATIO"] = (df["PRI_OVERDUE_ACCTS"] /
                            (df["PRI_NO_OF_ACCTS"] + 1))

# 2. Active account ratio
df["PRI_ACTIVE_RATIO"] = (df["PRI_ACTIVE_ACCTS"] /
                           (df["PRI_NO_OF_ACCTS"] + 1))

# 3. Net account health (new accounts minus delinquent)
df["NET_ACCTS_OMOM"] = (df["NEW_ACCTS_IN_LAST_SIX_MONTHS"] -
                         df["DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS"])

# 4. Total overdue ratio across PRI + SEC
df["TOTAL_NO_OF_ACCTS"]    = df["PRI_NO_OF_ACCTS"]    + df["SEC_NO_OF_ACCTS"]
df["TOTAL_ACTIVE_ACCTS"]   = df["PRI_ACTIVE_ACCTS"]   + df["SEC_ACTIVE_ACCTS"]
df["TOTAL_OVERDUE_ACCTS"]  = df["PRI_OVERDUE_ACCTS"]  + df["SEC_OVERDUE_ACCTS"]
df["TOTAL_CURRENT_BALANCE"]= df["PRI_CURRENT_BALANCE"] + df["SEC_CURRENT_BALANCE"]
df["TOTAL_DISBURSED_AMOUNT"]= df["PRI_DISBURSED_AMOUNT"] + df["SEC_DISBURSED_AMOUNT"]
df["TOTAL_OVERDUE_RATIO"]  = (df["TOTAL_OVERDUE_ACCTS"] /
                               (df["TOTAL_NO_OF_ACCTS"] + 1))

# 5. Loan-to-total-debt ratio
df["LOAN_TO_TOTAL_DEBT"] = (df["DISBURSED_AMOUNT"] /
                              (df["TOTAL_DISBURSED_AMOUNT"] + df["DISBURSED_AMOUNT"] + 1))

# 6. DOB components
df["DOB_YEAR"]  = df["DATE_OF_BIRTH"].dt.year
df["DOB_MONTH"] = df["DATE_OF_BIRTH"].dt.month
df["DOB_WEEK"]  = df["DATE_OF_BIRTH"].dt.isocalendar().week.astype(int)

# 7. Has PRI balance flag
df["HAS_PRI_BALANCE"] = (df["PRI_CURRENT_BALANCE"] > 0).astype(int)

# 8. Credit utilization proxy
df["CREDIT_UTILIZATION"] = (df["PRI_CURRENT_BALANCE"] /
                              (df["PRI_DISBURSED_AMOUNT"] + 1))

# 9. Recent stress indicator
df["RECENT_STRESS"] = (df["NO_OF_INQUIRIES"] +
                        df["DELINQUENT_ACCTS_IN_LAST_SIX_MONTHS"])

# 10. Account health score (composite)
df["CREDIT_HEALTH"] = (
    df["PRI_ACTIVE_ACCTS"] * 0.2 +
    (df["HAS_PRI_BALANCE"]) * 0.3 +
    (1 - df["PRI_OVERDUE_RATIO"]) * 0.4
)

new_features = ["TOTAL_NO_OF_ACCTS","TOTAL_ACTIVE_ACCTS","TOTAL_OVERDUE_ACCTS",
                 "NET_ACCTS_OMOM","TOTAL_CURRENT_BALANCE","TOTAL_DISBURSED_AMOUNT",
                 "TOTAL_OVERDUE_RATIO","LOAN_TO_TOTAL_DEBT"]
print("Features สร้างแล้ว:")
display(df[new_features].describe().round(2))


In [ ]:
# Default rate ของ interaction features
from sklearn.model_selection import StratifiedKFold, cross_val_score
from lightgbm import LGBMClassifier

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline (ไม่มี new features)
base_feats = [c for c in df.select_dtypes(include=["number","bool"]).columns
              if c not in ("LOAN_DEFAULT","UNIQUEID")]
X_base = df[base_feats].fillna(0)
y      = df["LOAN_DEFAULT"]

auc_baseline = cross_val_score(
    LGBMClassifier(scale_pos_weight=y.eq(0).sum()/y.sum(), random_state=42, verbose=-1),
    X_base, y, cv=cv, scoring="roc_auc"
).mean()

# With new features
X_new = df[base_feats + new_features].fillna(0)
auc_new = cross_val_score(
    LGBMClassifier(scale_pos_weight=y.eq(0).sum()/y.sum(), random_state=42, verbose=-1),
    X_new, y, cv=cv, scoring="roc_auc"
).mean()

print(f"CV AUC baseline:          {auc_baseline:.4f}")
print(f"CV AUC with new features: {auc_new:.4f}")
print(f"Improvement:              {auc_new - auc_baseline:+.4f}")


## 🔧 Outlier Detection & Log Transform

In [ ]:
# IQR outlier count per column
from scipy import stats as scipy_stats

num_cols2 = df.select_dtypes(include="number").columns.tolist()
num_cols2 = [c for c in num_cols2 if c not in ("UNIQUEID","LOAN_DEFAULT")]

Q1  = df[num_cols2].quantile(0.25)
Q3  = df[num_cols2].quantile(0.75)
IQR = Q3 - Q1
outlier_count = ((df[num_cols2] < Q1 - 1.5*IQR) | (df[num_cols2] > Q3 + 1.5*IQR)).sum()
print(outlier_count.sort_values(ascending=False).head(15))


In [ ]:
# Log transform comparison for DISBURSED_AMOUNT
skewed_cols = ["NEW_ACCTS_IN_LAST_SIX_MONTHS","PRI_CURRENT_BALANCE",
               "PRI_SANCTIONED_AMOUNT","PRIMARY_INSTAL_AMT","DISBURSED_AMOUNT","ASSET_COST"]

original_skew = df[skewed_cols].skew()
log_skew      = np.log1p(df[skewed_cols].clip(lower=0)).skew()
print(pd.DataFrame({"original": original_skew, "after_log": log_skew}).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
col = "DISBURSED_AMOUNT"
df[col].hist(bins=50, ax=axes[0]);               axes[0].set_title(f"Original (skew={df[col].skew():.2f})")
np.log1p(df[col]).hist(bins=50, ax=axes[1]);     axes[1].set_title(f"log1p (skew={np.log1p(df[col]).skew():.2f})")
np.sqrt(df[col]).hist(bins=50, ax=axes[2]);      axes[2].set_title(f"sqrt (skew={np.sqrt(df[col]).skew():.2f})")
plt.tight_layout(); plt.show()


## 🔵 Age-at-Disbursal Clustering

In [ ]:
from sklearn.cluster import KMeans

k_range = range(2, 11)
inertia_values = []
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(df[["AGE_AT_DISBURSAL"]].dropna())
    inertia_values.append(km.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(list(k_range), inertia_values, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Plot for AGE_AT_DISBURSAL")
plt.tight_layout(); plt.show()


In [ ]:
# FIX: sklearn version-safe silhouette score
from sklearn.metrics import silhouette_score

k_range_sil = range(2, 11)
sil_scores = []
for k in k_range_sil:
    km = KMeans(n_clusters=k, random_state=42)
    labels = km.fit_predict(df[["AGE_AT_DISBURSAL"]].dropna())
    score = silhouette_score(df[["AGE_AT_DISBURSAL"]].dropna(), labels)
    sil_scores.append(score)
    print(f"k={k}: silhouette={score:.4f}")

optimal_k = k_range_sil[np.argmax(sil_scores)]
print(f"\nOptimal k = {optimal_k}")


In [ ]:
# Apply clustering
km_final = KMeans(n_clusters=optimal_k, random_state=42)
df["AGE_AT_DISBURSAL_CLUSTER"] = km_final.fit_predict(df[["AGE_AT_DISBURSAL"]].fillna(df["AGE_AT_DISBURSAL"].median()))

# Order clusters by mean age
age_order = df.groupby("AGE_AT_DISBURSAL_CLUSTER")["AGE_AT_DISBURSAL"].mean().sort_values().index
cluster_rename = {old: new for new, old in enumerate(age_order)}
df["AGE_CLUSTER_ORDERED"] = df["AGE_AT_DISBURSAL_CLUSTER"].map(cluster_rename)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for cluster in sorted(df["AGE_CLUSTER_ORDERED"].unique()):
    subset = df[df["AGE_CLUSTER_ORDERED"] == cluster]
    axes[0].hist(subset["AGE_AT_DISBURSAL"], bins=20, alpha=0.6, label=f"Cluster {cluster}")
axes[0].set_title("Age Distribution per Cluster"); axes[0].legend()

df.groupby("AGE_CLUSTER_ORDERED")["LOAN_DEFAULT"].mean().plot(kind="bar", ax=axes[1], color="tomato")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(1))
axes[1].set_title("Default Rate per Cluster")

plt.tight_layout(); plt.show()
display(df.groupby("AGE_CLUSTER_ORDERED")["AGE_AT_DISBURSAL"].describe().round(1))


## 🪪 Identity Document Analysis

In [ ]:
id_flags = ["AADHAR_FLAG","PAN_FLAG","VOTERID_FLAG","DRIVING_FLAG","PASSPORT_FLAG"]

# Count IDs per borrower
df["ID_COUNT"] = df[id_flags].apply(lambda x: (x == 1).astype(int)).sum(axis=1)
print(df["ID_COUNT"].value_counts().sort_index())

# Default rate by ID_COUNT
dr_id = df.groupby("ID_COUNT")["LOAN_DEFAULT"].mean()
dr_id.plot(kind="bar", title="Default Rate by Number of ID Documents", color="tomato")
plt.ylabel("Default Rate"); plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1))
plt.tight_layout(); plt.show()

# Wilson CI
from statsmodels.stats.proportion import proportion_confint
results = []
for count in sorted(df["ID_COUNT"].unique()):
    subset   = df[df["ID_COUNT"] == count]["LOAN_DEFAULT"]
    n        = len(subset)
    defaults = subset.sum()
    rate     = defaults / n
    lower, upper = proportion_confint(defaults, n, alpha=0.05, method="wilson")
    results.append({"ID_COUNT": count, "n": n, "default_rate": rate, "CI_lower": lower, "CI_upper": upper})
display(pd.DataFrame(results).round(3))

df["HAS_EXTRA_ID"] = (df["ID_COUNT"] > 1).astype(int)
print("Default rate by HAS_EXTRA_ID:")
print(df.groupby("HAS_EXTRA_ID")["LOAN_DEFAULT"].mean())


## 💰 Credit Balance & LTV Analysis

In [ ]:
# LTV: no data error rows
print(f"LTV > 100: {(df['LTV'] > 100).sum()} rows")

# PRI_CURRENT_BALANCE: negative values check
print(f"\nPRI_CURRENT_BALANCE < 0: {(df['PRI_CURRENT_BALANCE'] < 0).sum()} rows")
print(f"PRI_CURRENT_BALANCE = 0: {(df['PRI_CURRENT_BALANCE'] == 0).sum()} rows")

df["PRI_CURRENT_BALANCE_LOG"] = np.log1p(df["PRI_CURRENT_BALANCE"].clip(lower=0))

print("\nDefault rate by HAS_PRI_BALANCE:")
print(df.groupby("HAS_PRI_BALANCE")["LOAN_DEFAULT"].mean())


In [ ]:
# Has overdue flag
df["HAS_OVERDUE"] = (df["PRI_OVERDUE_ACCTS"] > 0).astype(int)
print("Default rate — HAS_OVERDUE:")
print(df.groupby("HAS_OVERDUE")["LOAN_DEFAULT"].mean())

# MOBILENO_AVL_FLAG — always 1 → drop in modeling
print(f"\nMOBILENO_AVL_FLAG unique: {df['MOBILENO_AVL_FLAG'].unique()}")
print("  → Zero variance, drop in modeling")


## 📋 Credit History Trend Analysis

In [ ]:
df["HAS_CREDIT_HISTORY"] = (df["AVERAGE_ACCT_AGE_MON"] > 0).astype(int)
print("Default rate by HAS_CREDIT_HISTORY:")
print(df.groupby("HAS_CREDIT_HISTORY")["LOAN_DEFAULT"].mean())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in zip(
    axes,
    ["AVERAGE_ACCT_AGE_MON","CREDIT_HISTORY_LEN_MON"],
    ["Default Rate by AVERAGE_ACCT_AGE_MON","Default Rate by CREDIT_HISTORY_LEN_MON"]
):
    has_hist = df[df[col] > 0].copy()
    has_hist["bin"] = pd.qcut(has_hist[col], q=10, duplicates="drop")
    has_hist.groupby("bin")["LOAN_DEFAULT"].mean().plot(kind="bar", ax=ax, rot=45)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
    ax.set_title(title)
    ax.set_ylabel("Default Rate")

plt.tight_layout(); plt.show()


## ⚙️ Preprocessing Pipeline

In [ ]:
from sklearn.model_selection import train_test_split

# Feature / target
X = df.drop(columns=["LOAN_DEFAULT"])
y = df["LOAN_DEFAULT"]

# 80/20 stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train default rate: {y_train.mean():.3f}")
print(f"Test default rate:  {y_test.mean():.3f}")


In [ ]:
# Columns to drop
drop_cols = [
    "MOBILENO_AVL_FLAG",       # zero variance
    "UNIQUEID",                 # identifier
    "AVERAGE_ACCT_AGE",        # superseded by AVERAGE_ACCT_AGE_MON
    "CREDIT_HISTORY_LENGTH",   # superseded by CREDIT_HISTORY_LEN_MON
    # Multicollinearity: DISBURSED ≈ SANCTIONED
    "PRI_SANCTIONED_AMOUNT",
    "SEC_SANCTIONED_AMOUNT",
]
X_train = X_train.drop(columns=drop_cols, errors="ignore")
X_test  = X_test.drop(columns=drop_cols, errors="ignore")


In [ ]:
import category_encoders as ce

# Categorical columns to encode
cat_cols_enc = ["EMPLOYMENT_TYPE","PERFORM_CNS_SCORE_DESCRIPTION"]

# High-cardinality cols → Target Encoding
high_cardinality_cols = ["BRANCH_ID","SUPPLIER_ID","MANUFACTURER_ID",
                          "CURRENT_PINCODE_ID","STATE_ID","EMPLOYEE_CODE_ID"]

# Binary cols already 0/1 — keep as-is
binary_cols = ["AADHAR_FLAG","PAN_FLAG","VOTERID_FLAG","DRIVING_FLAG",
               "PASSPORT_FLAG","HAS_CREDIT_HISTORY","HAS_PRI_BALANCE",
               "HAS_OVERDUE","HAS_EXTRA_ID","AGE_CLUSTER_ORDERED"]

# CNS ordinal map
cns_risk_map = {
    "M-Very High Risk":13, "L-Very High Risk":12, "K-High Risk":11,
    "J-High Risk":10, "I-Medium Risk":9, "H-Medium Risk":8,
    "G-Low Risk":7, "F-Low Risk":6, "E-Low Risk":5, "D-Very Low Risk":4,
    "C-Very Low Risk":3, "A-Very Low Risk":2, "B-Very Low Risk":1,
    "No Bureau History Available":0,
    "Not Scored: Sufficient History Not Available":0,
    "Not Scored: No Activity seen on the customer (Inactive)":0,
    "Not Scored: No Updates available in last 36 months":0,
    "Not Scored: Not Enough Info available on the customer":0,
    "Not Scored: Only a Guarantor":0,
    "Not Scored: More than 50 active Accounts found":0,
}

if "PERFORM_CNS_SCORE_DESCRIPTION" in X_train.columns:
    X_train["CNS_RISK_ORDINAL"] = X_train["PERFORM_CNS_SCORE_DESCRIPTION"].map(cns_risk_map)
    X_test["CNS_RISK_ORDINAL"]  = X_test["PERFORM_CNS_SCORE_DESCRIPTION"].map(cns_risk_map)

if "EMPLOYMENT_TYPE" in X_train.columns:
    X_train = pd.get_dummies(X_train, columns=["EMPLOYMENT_TYPE"], drop_first=True)
    X_test  = pd.get_dummies(X_test,  columns=["EMPLOYMENT_TYPE"], drop_first=True)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

# Target Encoding for high-cardinality cols
valid_target_cols = [c for c in high_cardinality_cols if c in X_train.columns]
if valid_target_cols:
    te = ce.TargetEncoder(cols=valid_target_cols, smoothing=10)
    X_train[valid_target_cols] = te.fit_transform(X_train[valid_target_cols], y_train)
    X_test[valid_target_cols]  = te.transform(X_test[valid_target_cols])

# Drop original description column
cols_to_drop = [c for c in ["PERFORM_CNS_SCORE_DESCRIPTION","DATE_OF_BIRTH",
                              "DISBURSAL_DATE","AVERAGE_ACCT_AGE","CREDIT_HISTORY_LENGTH",
                              "DISBURSAL_MONTH"] if c in X_train.columns]
X_train = X_train.drop(columns=cols_to_drop, errors="ignore")
X_test  = X_test.drop(columns=cols_to_drop, errors="ignore")

print(f"X_train shape: {X_train.shape}")
print(f"X_test  shape: {X_test.shape}")


In [ ]:
from sklearn.preprocessing import RobustScaler

# RobustScaler — outlier-robust, ไม่ distort extreme values
scaler = RobustScaler()

num_cols_scale = X_train.select_dtypes(include="number").columns.tolist()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[num_cols_scale] = scaler.fit_transform(X_train[num_cols_scale])
X_test_scaled[num_cols_scale]  = scaler.transform(X_test[num_cols_scale])

# Class imbalance weight
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight = {scale_pos_weight:.2f}")


## ⚖️ SMOTE Oversampling (No Leakage)

> **Critical:** SMOTE ต้อง apply เฉพาะ train set เท่านั้น ห้ามรวม test set


In [ ]:
# Select only numeric columns for SMOTE (interpolation)
from imblearn.over_sampling import SMOTE

X_train_numeric = X_train.select_dtypes(include=["number","bool"])
X_test_numeric  = X_test.select_dtypes(include=["number","bool"])

# Fill NaN before SMOTE
X_train_numeric = X_train_numeric.fillna(X_train_numeric.median())
X_test_numeric  = X_test_numeric.fillna(X_train_numeric.median())  # use train median

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_numeric, y_train)

print(f"Original shape:    {X_train_numeric.shape}")
print(f"Resampled shape:   {X_train_resampled.shape}")
print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_resampled).value_counts().to_dict()}")


## 🤖 Model Comparison (Baseline)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, f1_score, classification_report
import pandas as pd

# FIX: convert numeric types only, no datetime columns
X_train_fin = X_train_numeric.select_dtypes(include=["number","bool"])
X_test_fin  = X_test_numeric.select_dtypes(include=["number","bool"])

models = {
    "Dummy": DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic Regression": LogisticRegression(
        class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", max_features="sqrt",
        max_depth=10, random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
    ),
    "LightGBM": LGBMClassifier(
        scale_pos_weight=scale_pos_weight,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42,
        verbose=-1,
        n_jobs=-1,
    ),
    "CatBoost": CatBoostClassifier(
        scale_pos_weight=scale_pos_weight,
        iterations=300,
        learning_rate=0.05,
        depth=6,
        random_seed=42,
        verbose=0,
        eval_metric="AUC",
    ),
}

summary = []
for name, model in models.items():
    print(f"Training {name}...")
    # Logistic Regression ใช้ scaled features
    X_tr = X_train_scaled[num_cols_scale] if name == "Logistic Regression" else X_train_fin
    X_te = X_test_scaled[num_cols_scale]  if name == "Logistic Regression" else X_test_fin
    # Align columns
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:,1]
    summary.append({
        "Model"   : name,
        "AUC-ROC" : round(roc_auc_score(y_test, y_prob), 4),
        "F1"      : round(f1_score(y_test, y_pred), 4),
        "Recall@1": round(f1_score(y_test, y_pred, pos_label=1, average="binary"), 4),
        "Precision@1": round(f1_score(y_test, y_pred, pos_label=1, average="binary"), 4),
    })

result_df = pd.DataFrame(summary).sort_values("AUC-ROC", ascending=False)
print("\n")
display(result_df)


## ⚖️ Imbalance Handling Comparison

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, classification_report

# Use XGBoost as base model — compare 3 strategies
strategies = {}

# 1. Baseline (scale_pos_weight only)
xgb_base = XGBClassifier(scale_pos_weight=scale_pos_weight,
                           n_estimators=300, learning_rate=0.05, max_depth=6,
                           subsample=0.8, colsample_bytree=0.8, random_state=42,
                           eval_metric="logloss", n_jobs=-1)
xgb_base.fit(X_train_fin, y_train)
strategies["Weighted"] = xgb_base

# 2. SMOTE (no leakage — applied on train only)
xgb_smote = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                            subsample=0.8, colsample_bytree=0.8, random_state=42,
                            eval_metric="logloss", n_jobs=-1)
xgb_smote.fit(X_train_resampled, y_train_resampled)
strategies["SMOTE"] = xgb_smote

for strat_name, model in strategies.items():
    y_pred = model.predict(X_test_fin)
    y_prob = model.predict_proba(X_test_fin)[:,1]
    print(f"{'='*40}")
    print(f"Method: {strat_name}")
    print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))


## 🎯 Threshold Tuning (NEW)

ปรับ decision threshold จาก 0.5 → optimal สำหรับ imbalanced data


In [ ]:
from sklearn.metrics import (precision_recall_curve, roc_curve,
                              f1_score, precision_score, recall_score)

# Use best model (XGBoost weighted)
y_prob_best = xgb_base.predict_proba(X_test_fin)[:,1]

# Precision-Recall curve
precision_arr, recall_arr, thresholds_pr = precision_recall_curve(y_test, y_prob_best)

# F1 at each threshold
f1_scores = 2 * precision_arr[:-1] * recall_arr[:-1] / (precision_arr[:-1] + recall_arr[:-1] + 1e-9)
best_idx  = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(thresholds_pr, precision_arr[:-1], label="Precision", color="steelblue")
axes[0].plot(thresholds_pr, recall_arr[:-1],    label="Recall",    color="tomato")
axes[0].plot(thresholds_pr, f1_scores,          label="F1",        color="green", linewidth=2)
axes[0].axvline(best_threshold, linestyle="--", color="gray", label=f"Best threshold={best_threshold:.3f}")
axes[0].set_xlabel("Threshold"); axes[0].set_ylabel("Score")
axes[0].set_title("Precision / Recall / F1 vs Threshold")
axes[0].legend()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_prob_best)
axes[1].plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, y_prob_best):.4f}")
axes[1].plot([0,1],[0,1],"--", color="gray")
axes[1].set_xlabel("FPR"); axes[1].set_ylabel("TPR")
axes[1].set_title("ROC Curve"); axes[1].legend()

plt.tight_layout(); plt.show()

# Compare at default threshold vs optimal threshold
for thresh, label in [(0.5, "Default (0.5)"), (best_threshold, f"Optimal ({best_threshold:.3f})")]:
    y_pred_t = (y_prob_best >= thresh).astype(int)
    print(f"\n--- {label} ---")
    print(f"F1:        {f1_score(y_test, y_pred_t):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred_t, zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred_t):.4f}")
    print(classification_report(y_test, y_pred_t))


## 📌 Feature Importance (XGBoost Gain)

In [ ]:
from xgboost import plot_importance

plt.figure(figsize=(10, 8))
plot_importance(xgb_base, importance_type="gain", max_num_features=20)
plt.title("XGBoost Feature Importance (Gain)")
plt.tight_layout(); plt.show()


In [ ]:
# LightGBM importance
import pandas as pd

lgb_model = strategies.get("Weighted", xgb_base)
if hasattr(lgb_model, "feature_importances_"):
    fi = pd.Series(lgb_model.feature_importances_,
                   index=X_train_fin.columns).sort_values(ascending=False).head(20)
    fi.plot(kind="barh", figsize=(8,8), title="Feature Importances")
    plt.tight_layout(); plt.show()


## 🔍 SHAP Values — Model Interpretability (NEW)

SHAP อธิบายว่า feature แต่ละตัวส่งผลต่อการ default อย่างไร


In [ ]:
try:
    import shap

    explainer  = shap.TreeExplainer(xgb_base)
    shap_vals  = explainer.shap_values(X_test_fin.fillna(0).head(2000))

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_vals, X_test_fin.fillna(0).head(2000),
                      plot_type="bar", show=False, max_display=15)
    plt.title("SHAP Feature Importance (mean |SHAP|)")
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(10, 9))
    shap.summary_plot(shap_vals, X_test_fin.fillna(0).head(2000),
                      show=False, max_display=15)
    plt.title("SHAP Beeswarm Plot")
    plt.tight_layout(); plt.show()

except ImportError:
    print("SHAP not installed. Run: pip install shap")
    print("ข้ามขั้นตอนนี้ไปก่อน")


## 📐 Multicollinearity Check — VIF (NEW)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [c for c in X_train_fin.columns if X_train_fin[c].nunique() > 2]
X_vif = X_train_fin[vif_cols].fillna(0)

# ลด columns เพื่อความเร็ว
vif_data = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values("VIF", ascending=False)

display(vif_data.head(20))
high_vif = vif_data[vif_data["VIF"] > 10]
print(f"\nFeatures with VIF > 10: {len(high_vif)}")
if not high_vif.empty:
    print(high_vif.to_string(index=False))


## 🔧 Hyperparameter Tuning — Optuna (LightGBM)

In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "scale_pos_weight": scale_pos_weight,
            "eval_metric": "logloss",
            "random_state": 42,
            "n_jobs": -1,
            "n_estimators":    trial.suggest_int("n_estimators", 100, 1000),
            "learning_rate":   trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "max_depth":       trial.suggest_int("max_depth", 3, 9),
            "min_child_weight":trial.suggest_int("min_child_weight", 1, 20),
            "subsample":       trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree":trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha":       trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            "reg_lambda":      trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            "gamma":           trial.suggest_float("gamma", 0, 5),
        }
        model = XGBClassifier(**params)
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        auc_scores = cross_val_score(model, X_train_fin, y_train,
                                      cv=cv, scoring="roc_auc", n_jobs=-1)
        return auc_scores.mean()

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
    print("Starting Optuna optimization...")
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    print(f"\nBest AUC: {study.best_value:.4f}")
    print(f"Best params: {study.best_params}")

    best_params = study.best_params
    best_params.update({
        "scale_pos_weight": scale_pos_weight,
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1,
        "early_stopping_rounds": 50,  # FIX: ใส่ใน constructor ได้ใน xgboost >= 2.0
    })

    xgb_tuned = XGBClassifier(**best_params)
    xgb_tuned.fit(X_train_fin, y_train,
                   eval_set=[(X_test_fin, y_test)],
                   verbose=False)

    y_prob_tuned = xgb_tuned.predict_proba(X_test_fin)[:,1]
    print(f"Before tuning AUC: {roc_auc_score(y_test, y_prob_best):.4f}")
    print(f"After tuning  AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")
    print(classification_report(y_test, (y_prob_tuned >= best_threshold).astype(int)))

except ImportError:
    print("Optuna not installed. Run: pip install optuna")


## 🏆 Final Model Summary & Recommendations

In [ ]:
# Summary table
print("="*60)
print("FINAL MODEL COMPARISON SUMMARY")
print("="*60)

final_models = {}

# Re-fit all models and record metrics
for name, model in models.items():
    X_tr = X_train_scaled[num_cols_scale] if name == "Logistic Regression" else X_train_fin
    X_te = X_test_scaled[num_cols_scale]  if name == "Logistic Regression" else X_test_fin
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

    if name == "Dummy":
        model.fit(X_tr, y_train)
    y_prob_i = model.predict_proba(X_te)[:,1]

    # Optimal threshold per model
    prec_i, rec_i, thresh_i = precision_recall_curve(y_test, y_prob_i)
    f1_i    = 2*prec_i[:-1]*rec_i[:-1]/(prec_i[:-1]+rec_i[:-1]+1e-9)
    best_t  = thresh_i[np.argmax(f1_i)] if len(thresh_i) > 0 else 0.5
    y_pred_i = (y_prob_i >= best_t).astype(int)

    final_models[name] = {
        "AUC-ROC"        : round(roc_auc_score(y_test, y_prob_i), 4),
        "F1 (optimal t)" : round(f1_score(y_test, y_pred_i), 4),
        "Recall (default)": round(recall_score(y_test, y_pred_i), 4),
        "Precision"      : round(precision_score(y_test, y_pred_i, zero_division=0), 4),
        "Best threshold" : round(best_t, 3),
    }

final_df = pd.DataFrame(final_models).T.sort_values("AUC-ROC", ascending=False)
display(final_df)


## 📝 Key Findings & Recommendations

### Key Findings
1. **LTV** คือ feature ที่สำคัญที่สุด — สัมพันธ์กับ default สูงสุด (Pearson r ≈ +0.10)
2. **CNS Risk Score** มีผลชัดเจน — M/L/K-Very High Risk มี default rate > 30%
3. **PRI_OVERDUE_RATIO** และ **NO_OF_INQUIRIES** เป็น signal ที่แข็งแกร่ง
4. **EMPLOYMENT_TYPE**: Self Employed มี default rate สูงกว่า Salaried (22.8% vs 20.3%)
5. **ID Documents**: ผู้มีเอกสารมาก (ID_COUNT=4) มี default rate สูงกว่า — ผิดปกติ ควรสืบสวน
6. **Geographic**: บาง STATE_ID มี default rate แตกต่างกันมาก → ใช้เป็น risk factor

### Model Recommendations
- **Production model**: LightGBM หรือ CatBoost (AUC ≈ 0.656)
- **Threshold**: ปรับ threshold ต่ำกว่า 0.5 เพื่อเพิ่ม recall (จับ default ได้มากขึ้น)
- **Business focus**: ถ้า cost of default > cost of false reject → lower threshold มากขึ้น

### Next Steps
- [ ] Ensemble/Stacking model
- [ ] Feature selection (VIF > 10 candidates)
- [ ] Deploy SHAP explanation ใน production scoring
- [ ] Monitor model drift รายเดือน
